Source: https://github.com/adityanagara/AHRF_project

# Parsing and visualizing the AHRF (Area Health Resource File)


This I-Python Notebook explores the AHRF (Area Health Resource File) data set available for download from the [HRSA (Health Resources and Service Administration)](http://ahrf.hrsa.gov/) website. 

## Introduction 
The county level AHRF contains 3230 records and 6920 variables. These variables include data on health care providers, health care facilities, health care facilities utilization, expenditures and demographic information at the county level. To optimize for speed, the API uses the multiprocessing module to parse the ASCII file. The API allows the user to select a set of variables to extract from the ASCII file and load it as a Pandas data frame. The description of the variables is given in the technical documentation which comes along with the zip file on downloading the data. The .sas file is used to obtain the variable names and locations of the variables on the ASCII file. A meta_data.csv is created as a look up table for variable name and location on the ASCII file once you call the object parse_AHRF_ascii(). We analyze a small subset of these variables at the county, state and county levels and discuss some correlations between these variables. 

In [1]:
import sys
import importlib

# Add the directory containing the AHRF parser script
sys.path.append(r'../AHRF_project_master/code/')
import pandas as pd
import numpy as np
import re
from matplotlib import pyplot as plt
import seaborn as sns 
%matplotlib inline
import AHRF_parser
importlib.reload(AHRF_parser)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## County Level Analysis
Let us begin by loading all of our variables as a Pandas data frame and look at the county averages of these variables over time. 

In [2]:
# Takes as input number of cores to use
ahrf_parser = AHRF_parser.parse_AHRF_ascii(num_cores = 8,
                                           ascii_file_path = "../AHRF_project_master/source/AHRF_2021-2022/DATA/ahrf2022.asc",
                                           sas_file_path="../AHRF_project_master/source/AHRF_2021-2022/DOC/AHRF2021-2022.sas")

In [3]:
# Determine a set of medical care providers variables to analyze
medical_providers_columns = ["FIPS State Code",
                              "FIPS County Code",
                              "State Name Abbreviation",
                               "County Name"]

medical_providers_columns.extend(["Census Population 2010","Census Population 2020"])
medical_providers_columns.extend(["Population Estimate {}".format(x) for x in range(2011,2020)])
medical_providers_columns.extend(["Population Estimate 2021"])
medical_providers_columns.extend(["Phys,Primary Care, Patient Care Non-Fed {}".format(x) for x in range(2010,2021)])
medical_providers_columns.extend(["Total Active M.D.s Non-Federal {}".format(x) for x in range(2010,2021)])
medical_providers_columns.extend(["Total Active D.O.s Non-Federal {}".format(x) for x in range(2010,2021)])

all_columns = medical_providers_columns

In [4]:
# # Determine a set of medical facilities utilization variables to analyze
utilization_columns = []
utilization_columns.extend(["Hospital Beds 2020", "Hospital Beds 2015", "Hospital Beds 2010",
                             "Short Term General Hosp Beds 2020", "Short Term General Hosp Beds 2015", "Short Term General Hosp Beds 2010",
                             "Inpatient Days in ST Gen Hosp 2020", "Inpatient Days in ST Gen Hosp 2015", "Inpatient Days in ST Gen Hosp 2010",
                             "Total Number Hospitals 2020","Total Number Hospitals 2015","Total Number Hospitals 2010",
                             "# Short Term General Hosps 2020", "# Short Term General Hosps 2015", "# Short Term General Hosps 2010",
                             "Hospital Admissions 2020","Hospital Admissions 2015","Hospital Admissions 2010",
                             "Short Term Gen Hosp Admissions 2020","Short Term Gen Hosp Admissions 2015","Short Term Gen Hosp Admissions 2010",
                             "Total Medicare Inpatient Days Short Term General Hospitals 2020","Total Medicare Inpatient Days Short Term General Hospitals 2015","Total Medicare Inpatient Days Short Term General Hospitals 2010",
                             "Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2020","Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2015","Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2010",
                             "Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2020","Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2015","Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2010",
                             "Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2020","Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2015","Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2010",
                             "Dist Hosp By 80+% Util Rate Short Term General Hospitals 2020","Dist Hosp By 80+% Util Rate Short Term General Hospitals 2015","Dist Hosp By 80+% Util Rate Short Term General Hospitals 2010"])
all_columns.extend(utilization_columns)

In [5]:
# Determine a set of demographics variables to analyze
demographics_columns = []
demographics_columns.extend(["% <65 without Health Insurance {}".format(x) for x in range(2015,2020)])
demographics_columns.extend(["% <65 without Health Insurance 2010"])
demographics_columns.extend(["Unemployment Rate, 16+ {}".format(x) for x in range(2015,2022)])
demographics_columns.extend(["Unemployment Rate, 16+ 2010"])
demographics_columns.extend(["Per Capita Personal Income {}".format(x) for x in range(2015,2021)])
demographics_columns.extend(["Per Capita Personal Income 2010"])
demographics_columns.extend(["Median Household Income {}".format(x) for x in range(2015,2021)])
demographics_columns.extend(["Median Household Income 2010"])
demographics_columns.extend(["Percent Persons in Poverty {}".format(x) for x in range(2015,2021)])
demographics_columns.extend(["Percent Persons in Poverty 2010"])
all_columns.extend(demographics_columns)

In [6]:
ahrf_frame = ahrf_parser.parse_ahrf_file_multicore(ahrf_columns = all_columns)
ahrf_frame.shape

Loading variables to a DataFrame...
[slice(0, 403, None), slice(403, 806, None), slice(806, 1209, None), slice(1209, 1612, None), slice(1612, 2015, None), slice(2015, 2418, None), slice(2418, 2821, None), slice(2821, 3230, None)]
Data frame saved at ../AHRF_project_master/DATA/ahrf_data.csv
Total time taken 16.5694 s


(3230, 120)

Let us define some helper functions that converts these the fields in the data frame to numerical fields. We also define a helper function to normalize a given variable by population. I chose the normalization by population as it gives a good estimate of density of resources as available to N people, the default value is N = 100000, but can be set as a parameter to the function.

In [7]:
def convert_to_numeric(data_frame):
    '''This function takes in a DataFrame and converst the given 
    columns to numeric value with NaN for any string that is not 
    a number.
    input: data frame
    
    output: data frame with all numerical fields conveterd to numerical data types
    '''
    for col in list(data_frame.keys()):
        if col not in ["FIPS State Code",
                              "FIPS County Code",
                              "State Name Abbreviation",
                               "County Name"]:
            data_frame[col] = pd.to_numeric(data_frame[col],errors = "coerce")
    return data_frame

In [8]:
ahrf_frame = convert_to_numeric(ahrf_frame)

In [9]:
ahrf_frame = ahrf_frame.rename(columns = {"Census Population 2010" : "Population Estimate 2010","Census Population 2020" : "Population Estimate 2020","County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame['County'] = ahrf_frame['County'].str.upper()
# Remove Guam, Puerto Rico and US Virgin Islands
#ahrf_frame = ahrf_frame[~ahrf_frame.FIPSStateCode.isin(["66","72","78"])]

In [10]:
ahrf_frame.columns

Index(['State_FIPS', 'County_FIPS', 'State', 'County',
       'Population Estimate 2010', 'Population Estimate 2020',
       'Population Estimate 2011', 'Population Estimate 2012',
       'Population Estimate 2013', 'Population Estimate 2014',
       ...
       'Median Household Income 2019', 'Median Household Income 2020',
       'Median Household Income 2010', 'Percent Persons in Poverty 2015',
       'Percent Persons in Poverty 2016', 'Percent Persons in Poverty 2017',
       'Percent Persons in Poverty 2018', 'Percent Persons in Poverty 2019',
       'Percent Persons in Poverty 2020', 'Percent Persons in Poverty 2010'],
      dtype='object', length=120)

In [11]:
ahrf_frame.head()

,State_FIPS,County_FIPS,State,County,Population Estimate 2010,Population Estimate 2020,Population Estimate 2011,Population Estimate 2012,Population Estimate 2013,Population Estimate 2014,...,Median Household Income 2019,Median Household Income 2020,Median Household Income 2010,Percent Persons in Poverty 2015,Percent Persons in Poverty 2016,Percent Persons in Poverty 2017,Percent Persons in Poverty 2018,Percent Persons in Poverty 2019,Percent Persons in Poverty 2020,Percent Persons in Poverty 2010
0,01,001,AL,AUTAUGA,54571.00,58805.00,55267.00,55514.00,55246.00,55395.00,...,58233.00,67565.00,53049.00,12.70,13.50,13.40,13.80,12.10,11.20,11.90
1,01,003,AL,BALDWIN,182265.00,231767.00,186717.00,190790.00,195540.00,200111.00,...,59871.00,71135.00,47618.00,12.90,11.70,10.10,9.80,10.10,8.90,13.30
2,01,005,AL,BARBOUR,27457.00,25223.00,27119.00,27201.00,27076.00,26887.00,...,35972.00,38866.00,33074.00,32.00,29.90,33.40,30.90,27.10,25.50,25.30
3,01,007,AL,BIBB,22915.00,22293.00,22766.00,22597.00,22512.00,22506.00,...,47918.00,50907.00,35472.00,22.20,20.10,20.20,21.80,20.30,17.80,20.90
4,01,009,AL,BLOUNT,57322.00,59134.00,57677.00,57826.00,57872.00,57719.00,...,52902.00,55203.00,42906.00,14.70,14.10,12.80,13.20,16.30,13.10,16.50


In [12]:
[col for col in ahrf_frame.columns if '2021' in col]

['Population Estimate 2021', 'Unemployment Rate, 16+ 2021']

In [13]:
ahrf_frame['State_FIPS'] = ahrf_frame['State_FIPS'].astype(str)

In [14]:
ahrf_frame['County_FIPS'].dtype

dtype('O')

In [15]:
# Defining our counties of interest
counties_of_interest = pd.read_csv('../data/hospitals_master.csv')[['State_FIPS','County_FIPS']].drop_duplicates()
counties_of_interest['State_FIPS'] = counties_of_interest['State_FIPS'].astype(int).astype(str).str.zfill(2)
counties_of_interest['County_FIPS'] = counties_of_interest['County_FIPS'].fillna('').astype(str).str.split('.').str[0]
counties_of_interest['County_FIPS'] = counties_of_interest['County_FIPS'].str.zfill(5)
counties_of_interest['County_FIPS'] = counties_of_interest['County_FIPS'].str[2:]

In [16]:
counties_of_interest.dtypes

State_FIPS     object
County_FIPS    object
dtype: object

In [17]:
counties_of_interest.shape

(765, 2)

In [18]:
counties_of_interest.head()

,State_FIPS,County_FIPS
0,22,113
1,27,053
2,04,013
6,35,006
7,48,201


In [19]:
ahrf_frame = ahrf_frame.merge(counties_of_interest, how='right', on=['State_FIPS','County_FIPS'])

In [20]:
ahrf_frame

,State_FIPS,County_FIPS,State,County,Population Estimate 2010,Population Estimate 2020,Population Estimate 2011,Population Estimate 2012,Population Estimate 2013,Population Estimate 2014,...,Median Household Income 2019,Median Household Income 2020,Median Household Income 2010,Percent Persons in Poverty 2015,Percent Persons in Poverty 2016,Percent Persons in Poverty 2017,Percent Persons in Poverty 2018,Percent Persons in Poverty 2019,Percent Persons in Poverty 2020,Percent Persons in Poverty 2010
0,22,113,LA,VERMILION,57999.00,57359.00,58276.00,58723.00,59253.00,59616.00,...,50193.00,52709.00,43498.00,18.10,19.60,17.00,18.30,17.00,15.40,17.50
1,27,053,MN,HENNEPIN,1152425.00,1281565.00,1168431.00,1184576.00,1198778.00,1212064.00,...,82323.00,81772.00,59252.00,10.90,10.90,10.50,10.30,9.70,9.30,13.70
2,04,013,AZ,MARICOPA,3817117.00,4420568.00,3880244.00,3942169.00,4009412.00,4087191.00,...,68634.00,71799.00,50424.00,16.30,15.00,13.50,12.30,12.20,11.60,16.60
3,35,006,NM,CIBOLA,27213.00,27172.00,27658.00,27334.00,27335.00,27349.00,...,40436.00,42705.00,33782.00,29.20,26.90,30.10,28.60,25.50,25.10,27.60
4,48,201,TX,HARRIS,4092459.00,4731145.00,4180894.00,4253700.00,4336853.00,4441370.00,...,61638.00,61906.00,50437.00,16.60,16.60,15.90,16.50,15.00,15.90,18.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,01,043,AL,CULLMAN,80406.00,87866.00,80536.00,80440.00,80811.00,81289.00,...,50897.00,51844.00,37948.00,19.40,14.90,13.80,14.50,12.20,12.50,19.20
761,36,065,NY,ONEIDA,234878.00,232125.00,234287.00,233556.00,233585.00,232871.00,...,56516.00,58696.00,46409.00,18.00,16.30,16.10,14.80,14.60,12.40,15.10
762,37,197,NC,YADKIN,38406.00,37214.00,38279.00,38084.00,38038.00,37792.00,...,50929.00,53154.00,41095.00,16.30,14.00,13.40,13.00,13.90,12.10,15.10
763,53,077,WA,YAKIMA,243231.00,256728.00,247141.00,246977.00,247044.00,247687.00,...,55674.00,56353.00,40503.00,19.10,18.20,18.10,16.50,16.70,14.80,23.90


In [21]:
ahrf_frame.isna().sum()

State_FIPS                          0
County_FIPS                         0
State                              28
County                             28
Population Estimate 2010           28
                                   ..
Percent Persons in Poverty 2017    41
Percent Persons in Poverty 2018    41
Percent Persons in Poverty 2019    41
Percent Persons in Poverty 2020    41
Percent Persons in Poverty 2010    41
Length: 120, dtype: int64

Reading in additional metrics from other years to get a full picture:

In [22]:
# Takes as input number of cores to use
ahrf_parser = AHRF_parser.parse_AHRF_ascii(num_cores = 8,
                                           ascii_file_path = "../AHRF_project_master/source/AHRF_2013-2014/DATA/ahrf2014.asc",
                                           sas_file_path="../AHRF_project_master/source/AHRF_2013-2014/DOC/ahrf2013-2014.sas")

# Determine a set of medical care providers variables to analyze
medical_providers_columns = ["FIPS State Code",
                              "FIPS County Code",
                              "State Name Abbreviation",
                               "County Name"]

all_columns = medical_providers_columns

# # Determine a set of medical facilities utilization variables to analyze
utilization_columns = []
utilization_columns.extend(["Hospital Beds 2011", 
                             "Short Term General Hosp Beds 2011", 
                             "Inpatient Days in ST Gen Hosp 2011",
                             "Total Number Hospitals 2011",
                             "# Short Term General Hosps 2011", 
                             "Hospital Admissions 2011",
                             "Short Term Gen Hosp Admissions 2011",
                             "Total Medicare Inpatient Days (ST Gen Hosps) 2011",
                             "Dist Hosp By 00 - 39% Util Rate STGEN 2011",
                             "Dist Hosp By 40 - 59% Util Rate STGEN 2011",
                             "Dist Hosp By 60 - 79% Util Rate STGEN 2011",
                             "Dist Hosp By 80+% Util Rate STGEN 2011"])
all_columns.extend(utilization_columns)

# Determine a set of demographics variables to analyze
demographics_columns = []
demographics_columns.extend(["% <65 without Health Insurance 2011"])
demographics_columns.extend(["Unemployment Rate, 16+ 2011"])
demographics_columns.extend(["Per Capita Personal Income 2011"])
demographics_columns.extend(["Median Household Income 2011"])
demographics_columns.extend(["Percent Persons in Poverty 2011"])
all_columns.extend(demographics_columns)

ahrf_frame_2011 = ahrf_parser.parse_ahrf_file_multicore(ahrf_columns = all_columns)

ahrf_frame_2011 = convert_to_numeric(ahrf_frame_2011)
ahrf_frame_2011 = ahrf_frame_2011.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2011['County'] = ahrf_frame_2011['County'].str.upper()

ahrf_frame_2011['State_FIPS'] = ahrf_frame_2011['State_FIPS'].astype(str)


Loading variables to a DataFrame...
[slice(0, 403, None), slice(403, 806, None), slice(806, 1209, None), slice(1209, 1612, None), slice(1612, 2015, None), slice(2015, 2418, None), slice(2418, 2821, None), slice(2821, 3230, None)]
 for Per Capita Personal Income 2011
Data frame saved at ../AHRF_project_master/DATA/ahrf_data.csv
Total time taken 5.2765 s







In [23]:
ahrf_frame_2011.head()

,State_FIPS,County_FIPS,State,County,Hospital Beds 2011,Short Term General Hosp Beds 2011,Inpatient Days in ST Gen Hosp 2011,Total Number Hospitals 2011,# Short Term General Hosps 2011,Hospital Admissions 2011,...,Total Medicare Inpatient Days (ST Gen Hosps) 2011,Dist Hosp By 00 - 39% Util Rate STGEN 2011,Dist Hosp By 40 - 59% Util Rate STGEN 2011,Dist Hosp By 60 - 79% Util Rate STGEN 2011,Dist Hosp By 80+% Util Rate STGEN 2011,% <65 without Health Insurance 2011,"Unemployment Rate, 16+ 2011",Per Capita Personal Income 2011,Median Household Income 2011,Percent Persons in Poverty 2011
0,01,001,AL,AUTAUGA,56.00,56.00,7124.00,1.00,1.00,2112.00,...,4402.00,1.00,0.00,0.00,0.00,13.90,8.00,NaN,48863.00,14.90
1,01,003,AL,BALDWIN,283.00,283.00,58788.00,3.00,3.00,14670.00,...,26661.00,1.00,1.00,1.00,0.00,16.60,8.10,NaN,50144.00,13.40
2,01,005,AL,BARBOUR,47.00,47.00,7102.00,1.00,1.00,1612.00,...,5493.00,0.00,1.00,0.00,0.00,18.90,11.40,NaN,30117.00,29.50
3,01,007,AL,BIBB,20.00,20.00,849.00,1.00,1.00,250.00,...,589.00,1.00,0.00,0.00,0.00,16.00,9.90,NaN,37347.00,22.20
4,01,009,AL,BLOUNT,40.00,40.00,4985.00,1.00,1.00,1367.00,...,3600.00,1.00,0.00,0.00,0.00,18.10,8.30,NaN,41940.00,14.90


In [24]:
# Takes as input number of cores to use
ahrf_parser = AHRF_parser.parse_AHRF_ascii(num_cores = 8,
                                           ascii_file_path = "../AHRF_project_master/source/AHRF_2014-2015/DATA/ahrf2015.asc",
                                           sas_file_path="../AHRF_project_master/source/AHRF_2014-2015/DOC/AHRF2015.sas")

# Determine a set of medical care providers variables to analyze
medical_providers_columns = ["FIPS State Code",
                              "FIPS County Code",
                              "State Name Abbreviation",
                               "County Name"]

all_columns = medical_providers_columns

# # Determine a set of medical facilities utilization variables to analyze
utilization_columns = []
utilization_columns.extend(["Hospital Beds 2012", 
                             "Short Term General Hosp Beds 2012", 
                             "Inpatient Days in ST Gen Hosp 2012",
                             "Total Number Hospitals 2012",
                             "# Short Term General Hosps 2012", 
                             "Hospital Admissions 2012",
                             "Short Term Gen Hosp Admissions 2012",
                             "Total Medicare Inpatient Days (ST Gen Hosps) 2012",
                             "Dist Hosp By 00 - 39% Util Rate STGEN 2012",
                             "Dist Hosp By 40 - 59% Util Rate STGEN 2012",
                             "Dist Hosp By 60 - 79% Util Rate STGEN 2012",
                             "Dist Hosp By 80+% Util Rate STGEN 2012"])
all_columns.extend(utilization_columns)

# Determine a set of demographics variables to analyze
demographics_columns = []
demographics_columns.extend(["% <65 without Health Insurance 2012"])
demographics_columns.extend(["Unemployment Rate, 16+ 2012"])
demographics_columns.extend(["Per Capita Personal Income 2012"])
demographics_columns.extend(["Median Household Income 2012"])
demographics_columns.extend(["Percent Persons in Poverty 2012"])
all_columns.extend(demographics_columns)

ahrf_frame_2012 = ahrf_parser.parse_ahrf_file_multicore(ahrf_columns = all_columns)

ahrf_frame_2012 = convert_to_numeric(ahrf_frame_2012)
ahrf_frame_2012 = ahrf_frame_2012.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2012['County'] = ahrf_frame_2012['County'].str.upper()

ahrf_frame_2012['State_FIPS'] = ahrf_frame_2012['State_FIPS'].astype(str)


Loading variables to a DataFrame...
[slice(0, 403, None), slice(403, 806, None), slice(806, 1209, None), slice(1209, 1612, None), slice(1612, 2015, None), slice(2015, 2418, None), slice(2418, 2821, None), slice(2821, 3230, None)]
Data frame saved at ../AHRF_project_master/DATA/ahrf_data.csv
Total time taken 4.0358 s

In [25]:
ahrf_frame_2012.columns

Index(['State_FIPS', 'County_FIPS', 'State', 'County', 'Hospital Beds 2012',
       'Short Term General Hosp Beds 2012',
       'Inpatient Days in ST Gen Hosp 2012', 'Total Number Hospitals 2012',
       '# Short Term General Hosps 2012', 'Hospital Admissions 2012',
       'Short Term Gen Hosp Admissions 2012',
       'Total Medicare Inpatient Days (ST Gen Hosps) 2012',
       'Dist Hosp By 00 - 39% Util Rate STGEN 2012',
       'Dist Hosp By 40 - 59% Util Rate STGEN 2012',
       'Dist Hosp By 60 - 79% Util Rate STGEN 2012',
       'Dist Hosp By 80+% Util Rate STGEN 2012',
       '% <65 without Health Insurance 2012', 'Unemployment Rate, 16+ 2012',
       'Per Capita Personal Income 2012', 'Median Household Income 2012',
       'Percent Persons in Poverty 2012'],
      dtype='object')

In [26]:
ahrf_frame_2012.head()

,State_FIPS,County_FIPS,State,County,Hospital Beds 2012,Short Term General Hosp Beds 2012,Inpatient Days in ST Gen Hosp 2012,Total Number Hospitals 2012,# Short Term General Hosps 2012,Hospital Admissions 2012,...,Total Medicare Inpatient Days (ST Gen Hosps) 2012,Dist Hosp By 00 - 39% Util Rate STGEN 2012,Dist Hosp By 40 - 59% Util Rate STGEN 2012,Dist Hosp By 60 - 79% Util Rate STGEN 2012,Dist Hosp By 80+% Util Rate STGEN 2012,% <65 without Health Insurance 2012,"Unemployment Rate, 16+ 2012",Per Capita Personal Income 2012,Median Household Income 2012,Percent Persons in Poverty 2012
0,01,001,AL,AUTAUGA,56.00,56.00,6937.00,1.00,1.00,2015.00,...,4323.00,1.00,0.00,0.00,0.00,12.80,6.50,NaN,51441.00,12.70
1,01,003,AL,BALDWIN,316.00,298.00,63275.00,4.00,3.00,16499.00,...,27891.00,1.00,0.00,2.00,0.00,15.80,6.80,NaN,48867.00,13.90
2,01,005,AL,BARBOUR,47.00,47.00,8635.00,1.00,1.00,1665.00,...,6550.00,0.00,1.00,0.00,0.00,17.50,11.20,NaN,30287.00,29.00
3,01,007,AL,BIBB,20.00,20.00,965.00,1.00,1.00,253.00,...,651.00,1.00,0.00,0.00,0.00,15.10,7.60,NaN,37392.00,21.50
4,01,009,AL,BLOUNT,40.00,40.00,4975.00,1.00,1.00,1232.00,...,3639.00,1.00,0.00,0.00,0.00,18.30,6.20,NaN,44225.00,16.20


In [27]:
# Takes as input number of cores to use
ahrf_parser = AHRF_parser.parse_AHRF_ascii(num_cores = 8,
                                           ascii_file_path = "../AHRF_project_master/source/AHRF_2015-2016/DATA/ahrf2016.asc",
                                           sas_file_path="../AHRF_project_master/source/AHRF_2015-2016/DOC/ahrf2015-16.sas")

# Determine a set of medical care providers variables to analyze
medical_providers_columns = ["FIPS State Code",
                              "FIPS County Code",
                              "State Name Abbreviation",
                               "County Name"]

all_columns = medical_providers_columns

# # Determine a set of medical facilities utilization variables to analyze
utilization_columns = []
utilization_columns.extend(["Hospital Beds  2013", 
                             "Short Term General Hosp Beds  2013", 
                             "Inpatient Days in ST Gen Hosp  2013",
                             "Total Number Hospitals  2013",
                             "# Short Term General Hosps  2013", 
                             "Hospital Admissions  2013",
                             "Short Term Gen Hosp Admissions  2013",
                             "Total Medicare Inpatient Days Short Term General Hospitals 2013",
                             "Dist Hosp By 00 - 39%  Util Rate Short Term General Hospitals 2013",
                             "Dist Hosp By 40 - 59%  Util Rate Short Term General Hospitals 2013",
                             "Dist Hosp By 60 - 79%  Util Rate Short Term General Hospitals 2013",
                             "Dist Hosp By 80+%  Util Rate Short Term General Hospitals 2013"])
all_columns.extend(utilization_columns)

# Determine a set of demographics variables to analyze
demographics_columns = []
demographics_columns.extend(["%  <65 without Health Insurance  2013"])
demographics_columns.extend(["Unemployment Rate, 16+     2013 "])
demographics_columns.extend(["Per Capita Personal Income  2013"])
demographics_columns.extend(["Per Capita Personal Income  2011"])
demographics_columns.extend(["Per Capita Personal Income  2012"])
demographics_columns.extend(["Median Household Income  2013"])
demographics_columns.extend(["Percent Persons in Poverty  2013"])
all_columns.extend(demographics_columns)

ahrf_frame_2013 = ahrf_parser.parse_ahrf_file_multicore(ahrf_columns = all_columns)

ahrf_frame_2013 = convert_to_numeric(ahrf_frame_2013)
ahrf_frame_2013 = ahrf_frame_2013.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2013['County'] = ahrf_frame_2013['County'].str.upper()

ahrf_frame_2013['State_FIPS'] = ahrf_frame_2013['State_FIPS'].astype(str)


Loading variables to a DataFrame...
[slice(0, 403, None), slice(403, 806, None), slice(806, 1209, None), slice(1209, 1612, None), slice(1612, 2015, None), slice(2015, 2418, None), slice(2418, 2821, None), slice(2821, 3230, None)]
Data frame saved at ../AHRF_project_master/DATA/ahrf_data.csv
Total time taken 4.0866 s


In [28]:
ahrf_frame_2013.head()

,State_FIPS,County_FIPS,State,County,Hospital Beds 2013,Short Term General Hosp Beds 2013,Inpatient Days in ST Gen Hosp 2013,Total Number Hospitals 2013,# Short Term General Hosps 2013,Hospital Admissions 2013,...,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2013,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2013,Dist Hosp By 80+% Util Rate Short Term General Hospitals 2013,% <65 without Health Insurance 2013,"Unemployment Rate, 16+ 2013",Per Capita Personal Income 2013,Per Capita Personal Income 2011,Per Capita Personal Income 2012,Median Household Income 2013,Percent Persons in Poverty 2013
0,01,001,AL,AUTAUGA,50.00,50.00,9680.00,1.00,1.00,2469.00,...,1.00,0.00,0.00,13.40,NaN,34843.00,32657.00,34043.00,51868.00,13.50
1,01,003,AL,BALDWIN,364.00,298.00,65999.00,4.00,3.00,17871.00,...,1.00,2.00,0.00,17.40,NaN,39100.00,38024.00,38548.00,47539.00,14.20
2,01,005,AL,BARBOUR,47.00,47.00,8489.00,1.00,1.00,1492.00,...,1.00,0.00,0.00,17.50,NaN,28559.00,26807.00,27771.00,30981.00,28.20
3,01,007,AL,BIBB,20.00,20.00,1082.00,1.00,1.00,251.00,...,0.00,0.00,0.00,15.10,NaN,25352.00,24180.00,25001.00,39781.00,23.10
4,01,009,AL,BLOUNT,40.00,40.00,4219.00,1.00,1.00,1087.00,...,0.00,0.00,0.00,17.60,NaN,29222.00,27220.00,28799.00,44392.00,17.20


In [29]:
# Takes as input number of cores to use
ahrf_parser = AHRF_parser.parse_AHRF_ascii(num_cores = 8,
                                           ascii_file_path = "../AHRF_project_master/source/AHRF_2016-2017/DATA/ahrf2017.asc",
                                           sas_file_path="../AHRF_project_master/source/AHRF_2016-2017/DOC/ahrf2016-17.sas")

# Determine a set of medical care providers variables to analyze
medical_providers_columns = ["FIPS State Code",
                              "FIPS County Code",
                              "State Name Abbreviation",
                               "County Name"]

all_columns = medical_providers_columns

# # Determine a set of medical facilities utilization variables to analyze
utilization_columns = []
utilization_columns.extend(["Hospital Beds 2014", 
                             "Short Term General Hosp Beds 2014", 
                             "Inpatient Days in ST Gen Hosp 2014",
                             "Total Number Hospitals 2014",
                             "# Short Term General Hosps 2014", 
                             "Hospital Admissions 2014",
                             "Short Term Gen Hosp Admissions 2014",
                             "Total Medicare Inpatient Days Short Term General Hospitals 2014",
                             "Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2014",
                             "Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2014",
                             "Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2014",
                             "Dist Hosp By 80+% Util Rate Short Term General Hospitals 2014"])
all_columns.extend(utilization_columns)

# Determine a set of demographics variables to analyze
demographics_columns = []
demographics_columns.extend(["% <65 without Health Insurance 2014"])
demographics_columns.extend(["Unemployment Rate, 16+ 2014"])
demographics_columns.extend(["Per Capita Personal Income 2014"])
demographics_columns.extend(["Median Household Income 2014"])
demographics_columns.extend(["Percent Persons in Poverty 2014"])
all_columns.extend(demographics_columns)

ahrf_frame_2014 = ahrf_parser.parse_ahrf_file_multicore(ahrf_columns = all_columns)

ahrf_frame_2014 = convert_to_numeric(ahrf_frame_2014)
ahrf_frame_2014 = ahrf_frame_2014.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2014['County'] = ahrf_frame_2014['County'].str.upper()

ahrf_frame_2014['State_FIPS'] = ahrf_frame_2014['State_FIPS'].astype(str)


Loading variables to a DataFrame...
[slice(0, 403, None), slice(403, 806, None), slice(806, 1209, None), slice(1209, 1612, None), slice(1612, 2015, None), slice(2015, 2418, None), slice(2418, 2821, None), slice(2821, 3230, None)]
Data frame saved at ../AHRF_project_master/DATA/ahrf_data.csv
Total time taken 5.7633 s


In [30]:
ahrf_frame_2014.head()

,State_FIPS,County_FIPS,State,County,Hospital Beds 2014,Short Term General Hosp Beds 2014,Inpatient Days in ST Gen Hosp 2014,Total Number Hospitals 2014,# Short Term General Hosps 2014,Hospital Admissions 2014,...,Total Medicare Inpatient Days Short Term General Hospitals 2014,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2014,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2014,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2014,Dist Hosp By 80+% Util Rate Short Term General Hospitals 2014,% <65 without Health Insurance 2014,"Unemployment Rate, 16+ 2014",Per Capita Personal Income 2014,Median Household Income 2014,Percent Persons in Poverty 2014
0,01,001,AL,AUTAUGA,50.00,50.00,10962.00,1.00,1.00,2535.00,...,7069.00,0.00,0.00,1.00,0.00,11.00,5.90,36419.00,54366.00,13.10
1,01,003,AL,BALDWIN,364.00,298.00,69449.00,4.00,3.00,18084.00,...,30507.00,0.00,1.00,2.00,0.00,16.10,6.10,39040.00,49626.00,13.00
2,01,005,AL,BARBOUR,47.00,47.00,9107.00,1.00,1.00,1639.00,...,7182.00,0.00,1.00,0.00,0.00,15.30,10.80,30449.00,34971.00,25.40
3,01,007,AL,BIBB,161.00,161.00,46807.00,1.00,1.00,671.00,...,4546.00,0.00,0.00,0.00,1.00,13.60,7.10,28314.00,39546.00,18.10
4,01,009,AL,BLOUNT,25.00,25.00,4939.00,1.00,1.00,1096.00,...,3954.00,0.00,1.00,0.00,0.00,16.50,6.10,31464.00,45567.00,17.50


In [31]:
# Takes as input number of cores to use
ahrf_parser = AHRF_parser.parse_AHRF_ascii(num_cores = 8,
                                           ascii_file_path = "../AHRF_project_master/source/AHRF_2017-2018/DATA/ahrf2018.asc",
                                           sas_file_path="../AHRF_project_master/source/AHRF_2017-2018/DOC/AHRF2017-18.sas")

# Determine a set of medical care providers variables to analyze
medical_providers_columns = ["FIPS State Code",
                              "FIPS County Code",
                              "State Name Abbreviation",
                               "County Name"]

all_columns = medical_providers_columns

# Determine a set of medical facilities utilization variables to analyze
utilization_columns = []
utilization_columns.extend(["Hospital Beds 2016", 
                             "Short Term General Hosp Beds 2016", 
                             "Inpatient Days in ST Gen Hosp 2016",
                             "Total Number Hospitals 2016",
                             "# Short Term General Hosps 2016", 
                             "Hospital Admissions 2016",
                             "Short Term Gen Hosp Admissions 2016",
                             "Total Medicare Inpatient Days Short Term General Hospitals 2016",
                             "Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2016",
                             "Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2016",
                             "Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2016",
                             "Dist Hosp By 80+% Util Rate Short Term General Hospitals 2016"])
all_columns.extend(utilization_columns)


ahrf_frame_2016 = ahrf_parser.parse_ahrf_file_multicore(ahrf_columns = all_columns)

ahrf_frame_2016 = convert_to_numeric(ahrf_frame_2016)
ahrf_frame_2016 = ahrf_frame_2016.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2016['County'] = ahrf_frame_2016['County'].str.upper()

ahrf_frame_2016['State_FIPS'] = ahrf_frame_2016['State_FIPS'].astype(str)


Loading variables to a DataFrame...
[slice(0, 403, None), slice(403, 806, None), slice(806, 1209, None), slice(1209, 1612, None), slice(1612, 2015, None), slice(2015, 2418, None), slice(2418, 2821, None), slice(2821, 3230, None)]
Data frame saved at ../AHRF_project_master/DATA/ahrf_data.csv
Total time taken 5.2952 s


In [32]:
ahrf_frame_2016.head()

,State_FIPS,County_FIPS,State,County,Hospital Beds 2016,Short Term General Hosp Beds 2016,Inpatient Days in ST Gen Hosp 2016,Total Number Hospitals 2016,# Short Term General Hosps 2016,Hospital Admissions 2016,Short Term Gen Hosp Admissions 2016,Total Medicare Inpatient Days Short Term General Hospitals 2016,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2016,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2016,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2016,Dist Hosp By 80+% Util Rate Short Term General Hospitals 2016
0,01,001,AL,AUTAUGA,50.00,50.00,12721.00,1.00,1.00,3200.00,3200.00,8301.00,0.00,0.00,1.00,0.00
1,01,003,AL,BALDWIN,364.00,298.00,71716.00,4.00,3.00,18931.00,17542.00,30121.00,0.00,1.00,2.00,0.00
2,01,005,AL,BARBOUR,47.00,47.00,6489.00,1.00,1.00,1297.00,1297.00,4851.00,1.00,0.00,0.00,0.00
3,01,007,AL,BIBB,180.00,180.00,45326.00,1.00,1.00,644.00,644.00,7001.00,0.00,0.00,1.00,0.00
4,01,009,AL,BLOUNT,25.00,25.00,5663.00,1.00,1.00,1057.00,1057.00,4609.00,0.00,0.00,1.00,0.00


In [33]:
# Takes as input number of cores to use
ahrf_parser = AHRF_parser.parse_AHRF_ascii(num_cores = 8,
                                           ascii_file_path = "../AHRF_project_master/source/AHRF_2018-2019/DATA/ahrf2019.asc",
                                           sas_file_path="../AHRF_project_master/source/AHRF_2018-2019/DOC/AHRF2018-19.sas")

# Determine a set of medical care providers variables to analyze
medical_providers_columns = ["FIPS State Code",
                              "FIPS County Code",
                              "State Name Abbreviation",
                               "County Name"]

all_columns = medical_providers_columns

# Determine a set of medical facilities utilization variables to analyze
utilization_columns = []
utilization_columns.extend(["Hospital Beds 2017", 
                             "Short Term General Hosp Beds 2017", 
                             "Inpatient Days in ST Gen Hosp 2017",
                             "Total Number Hospitals 2017",
                             "# Short Term General Hosps 2017", 
                             "Hospital Admissions 2017",
                             "Short Term Gen Hosp Admissions 2017",
                             "Total Medicare Inpatient Days Short Term General Hospitals 2017",
                             "Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2017",
                             "Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2017",
                             "Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2017",
                             "Dist Hosp By 80+% Util Rate Short Term General Hospitals 2017"])
all_columns.extend(utilization_columns)


ahrf_frame_2017 = ahrf_parser.parse_ahrf_file_multicore(ahrf_columns = all_columns)

ahrf_frame_2017 = convert_to_numeric(ahrf_frame_2017)
ahrf_frame_2017 = ahrf_frame_2017.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2017['County'] = ahrf_frame_2017['County'].str.upper()

ahrf_frame_2017['State_FIPS'] = ahrf_frame_2017['State_FIPS'].astype(str)


Loading variables to a DataFrame...
[slice(0, 403, None), slice(403, 806, None), slice(806, 1209, None), slice(1209, 1612, None), slice(1612, 2015, None), slice(2015, 2418, None), slice(2418, 2821, None), slice(2821, 3230, None)]
Data frame saved at ../AHRF_project_master/DATA/ahrf_data.csv
Total time taken 5.6558 s


In [34]:
ahrf_frame_2017.head()

,State_FIPS,County_FIPS,State,County,Hospital Beds 2017,Short Term General Hosp Beds 2017,Inpatient Days in ST Gen Hosp 2017,Total Number Hospitals 2017,# Short Term General Hosps 2017,Hospital Admissions 2017,Short Term Gen Hosp Admissions 2017,Total Medicare Inpatient Days Short Term General Hospitals 2017,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2017,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2017,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2017,Dist Hosp By 80+% Util Rate Short Term General Hospitals 2017
0,01,001,AL,AUTAUGA,55.00,55.00,14142.00,1.00,1.00,3428.00,3428.00,9066.00,0.00,0.00,1.00,0.00
1,01,003,AL,BALDWIN,348.00,298.00,69624.00,4.00,3.00,18667.00,17249.00,34488.00,0.00,1.00,2.00,0.00
2,01,005,AL,BARBOUR,47.00,47.00,6407.00,1.00,1.00,1259.00,1259.00,3245.00,1.00,0.00,0.00,0.00
3,01,007,AL,BIBB,180.00,180.00,47818.00,1.00,1.00,999.00,999.00,5414.00,0.00,0.00,1.00,0.00
4,01,009,AL,BLOUNT,25.00,25.00,4671.00,1.00,1.00,837.00,837.00,3411.00,0.00,1.00,0.00,0.00


In [35]:
ahrf_frame = ahrf_frame.merge(ahrf_frame_2011, how='left', on=['State_FIPS','County_FIPS','State','County'])
ahrf_frame = ahrf_frame.merge(ahrf_frame_2012, how='left', on=['State_FIPS','County_FIPS','State','County'])
ahrf_frame = ahrf_frame.merge(ahrf_frame_2013, how='left', on=['State_FIPS','County_FIPS','State','County'])
ahrf_frame = ahrf_frame.merge(ahrf_frame_2014, how='left', on=['State_FIPS','County_FIPS','State','County'])
ahrf_frame = ahrf_frame.merge(ahrf_frame_2016, how='left', on=['State_FIPS','County_FIPS','State','County'])
ahrf_frame = ahrf_frame.merge(ahrf_frame_2017, how='left', on=['State_FIPS','County_FIPS','State','County'])

In [36]:
ahrf_frame.shape

(765, 214)

In [37]:
ahrf_frame.columns

Index(['State_FIPS', 'County_FIPS', 'State', 'County',
       'Population Estimate 2010', 'Population Estimate 2020',
       'Population Estimate 2011', 'Population Estimate 2012',
       'Population Estimate 2013', 'Population Estimate 2014',
       ...
       'Inpatient Days in ST Gen Hosp 2017', 'Total Number Hospitals 2017',
       '# Short Term General Hosps 2017', 'Hospital Admissions 2017',
       'Short Term Gen Hosp Admissions 2017',
       'Total Medicare Inpatient Days Short Term General Hospitals 2017',
       'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2017',
       'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2017',
       'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2017',
       'Dist Hosp By 80+% Util Rate Short Term General Hospitals 2017'],
      dtype='object', length=214)

In [38]:
ahrf_frame = ahrf_frame.drop(columns=['Per Capita Personal Income  2011','Per Capita Personal Income  2012'])

In [39]:
# Takes as input number of cores to use
ahrf_parser = AHRF_parser.parse_AHRF_ascii(num_cores = 8,
                                           ascii_file_path = "../AHRF_project_master/source/AHRF_2020-2021/DATA/ahrf2021.asc",
                                           sas_file_path="../AHRF_project_master/source/AHRF_2020-2021/DOC/AHRF2020-2021.sas")

# Determine a set of medical care providers variables to analyze
medical_providers_columns = ["FIPS State Code",
                              "FIPS County Code",
                              "State Name Abbreviation",
                               "County Name"]

all_columns = medical_providers_columns

# Determine a set of medical facilities utilization variables to analyze
utilization_columns = []
utilization_columns.extend(["Hospital Beds 2019", 
                             "Short Term General Hosp Beds 2019", 
                             "Inpatient Days in ST Gen Hosp 2019",
                             "Total Number Hospitals 2019",
                             "# Short Term General Hosps 2019", 
                             "Hospital Admissions 2019",
                             "Short Term Gen Hosp Admissions 2019",
                             "Total Medicare Inpatient Days Short Term General Hospitals 2019",
                             "Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2019",
                             "Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2019",
                             "Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2019",
                             "Dist Hosp By 80+% Util Rate Short Term General Hospitals 2019"])
all_columns.extend(utilization_columns)


ahrf_frame_2019 = ahrf_parser.parse_ahrf_file_multicore(ahrf_columns = all_columns)

ahrf_frame_2019 = convert_to_numeric(ahrf_frame_2019)
ahrf_frame_2019 = ahrf_frame_2019.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2019['County'] = ahrf_frame_2019['County'].str.upper()

ahrf_frame_2019['State_FIPS'] = ahrf_frame_2019['State_FIPS'].astype(str)


Loading variables to a DataFrame...
[slice(0, 403, None), slice(403, 806, None), slice(806, 1209, None), slice(1209, 1612, None), slice(1612, 2015, None), slice(2015, 2418, None), slice(2418, 2821, None), slice(2821, 3230, None)]
Data frame saved at ../AHRF_project_master/DATA/ahrf_data.csv
Total time taken 3.5256 s


In [40]:
ahrf_frame_2019.head()

,State_FIPS,County_FIPS,State,County,Hospital Beds 2019,Short Term General Hosp Beds 2019,Inpatient Days in ST Gen Hosp 2019,Total Number Hospitals 2019,# Short Term General Hosps 2019,Hospital Admissions 2019,Short Term Gen Hosp Admissions 2019,Total Medicare Inpatient Days Short Term General Hospitals 2019,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2019,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2019,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2019,Dist Hosp By 80+% Util Rate Short Term General Hospitals 2019
0,01,001,AL,AUTAUGA,55.00,55.00,12509.00,1.00,1.00,2895.00,2895.00,8136.00,0.00,0.00,1.00,0.00
1,01,003,AL,BALDWIN,348.00,298.00,67477.00,4.00,3.00,18172.00,17394.00,29291.00,0.00,1.00,2.00,0.00
2,01,005,AL,BARBOUR,47.00,47.00,6412.00,1.00,1.00,1220.00,1220.00,3297.00,1.00,0.00,0.00,0.00
3,01,007,AL,BIBB,166.00,166.00,44394.00,1.00,1.00,1146.00,1146.00,6788.00,0.00,0.00,1.00,0.00
4,01,009,AL,BLOUNT,25.00,25.00,4535.00,1.00,1.00,839.00,839.00,3796.00,0.00,1.00,0.00,0.00


In [41]:
ahrf_frame = ahrf_frame.merge(ahrf_frame_2019, how='left', on=['State_FIPS','County_FIPS','State','County'])

In [42]:
# Determine a set of medical care providers variables to analyze
medical_providers_columns = {"fips_st":"FIPS State Code",
                              "fips_cnty":"FIPS County Code",
                              "st_name_abbrev":"State Name Abbreviation",
                               "cnty_name":"County Name"}

medical_providers_columns['popn_est_22']="Population Estimate 2022"
medical_providers_columns['phys_nf_prim_care_pc_exc_rsdt_21']="Phys,Primary Care, Patient Care Non-Fed 2021"
medical_providers_columns['md_nf_activ_21']="Total Active M.D.s Non-Federal 2021"
medical_providers_columns['do_nf_activ_21']="Total Active D.O.s Non-Federal 2021"

all_columns = medical_providers_columns

all_columns['pers_noins_lt65_pct_20']='% <65 without Health Insurance 2020'


list(all_columns)

# Determine a set of medical facilities utilization variables to analyze
all_columns['hosp_beds_21'] = 'Hospital Beds 2021'
all_columns['stgh_hosp_beds_21'] = 'Short Term General Hosp Beds 2021'
all_columns['stgh_inpat_dys_21'] = 'Inpatient Days in ST Gen Hosp 2021'
all_columns['hosp_21'] = 'Total Number Hospitals 2021'
all_columns['stgh_21'] = '# Short Term General Hosps 2021'
all_columns['hosp_adm_21'] = 'Hospital Admissions 2021'
all_columns['stgh_hosp_adm_21'] = 'Short Term Gen Hosp Admissions 2021'
all_columns['stgh_medcr_inpat_dys_21'] = 'Total Medicare Inpatient Days Short Term General Hospitals 2021'
all_columns['stgh_00_39_util_rate_pct_21'] = 'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2021'
all_columns['stgh_40_59_util_rate_pct_21'] = 'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2021'
all_columns['stgh_60_79_util_rate_pct_21'] = 'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2021'
all_columns['stgh_80_plus_util_rate_pct_21'] = 'Dist Hosp By 80+% Util Rate Short Term General Hospitals 2021'

# Determine a set of demographics variables to analyze
#all_columns.extend(["% <65 without Health Insurance 2010"])
all_columns['unemply_rate_ge16_22']="Unemployment Rate, 16+ 2022"
all_columns['per_cap_persnl_incom_21']="Per Capita Personal Income 2021"
all_columns['medn_hhi_acs_21']="Median Household Income 2021"
all_columns['pers_povty_pct_21']="Percent Persons in Poverty 2021"


ahrf_frame_2023 = pd.read_csv('../AHRF_project_master/source/AHRF_2022-2023/DATA/ahrf2023.csv')[list(all_columns)]
ahrf_frame_2023 = ahrf_frame_2023.rename(columns=all_columns)
ahrf_frame_2023 = convert_to_numeric(ahrf_frame_2023)
ahrf_frame_2023 = ahrf_frame_2023.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2023['County'] = ahrf_frame_2023['County'].str.upper()
ahrf_frame_2023['State_FIPS'] = ahrf_frame_2023['State_FIPS'].astype(str)
ahrf_frame_2023['County_FIPS'] = ahrf_frame_2023['County_FIPS'].astype(str)

ahrf_frame_2023.head()

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91901/3684631488.py:41: DtypeWarning: Columns (14,16,24) have mixed types. Specify dtype option on import or set low_memory=False.
  ahrf_frame_2023 = pd.read_csv('../AHRF_project_master/source/AHRF_2022-2023/DATA/ahrf2023.csv')[list(all_columns)]


,State_FIPS,County_FIPS,State,County,Population Estimate 2022,"Phys,Primary Care, Patient Care Non-Fed 2021",Total Active M.D.s Non-Federal 2021,Total Active D.O.s Non-Federal 2021,% <65 without Health Insurance 2020,Hospital Beds 2021,...,Short Term Gen Hosp Admissions 2021,Total Medicare Inpatient Days Short Term General Hospitals 2021,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2021,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2021,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2021,Dist Hosp By 80+% Util Rate Short Term General Hospitals 2021,"Unemployment Rate, 16+ 2022",Per Capita Personal Income 2021,Median Household Income 2021,Percent Persons in Poverty 2021
0,1,1,AL,AUTAUGA,59759.00,26.00,41.00,3.00,10.60,69.00,...,3023.00,9550.00,0.00,0.00,1.00,0.00,2.30,48347.00,62660.00,10.70
1,1,3,AL,BALDWIN,246435.00,150.00,468.00,45.00,10.90,405.00,...,19436.00,42583.00,0.00,1.00,2.00,0.00,2.40,54659.00,64346.00,10.80
2,1,5,AL,BARBOUR,24706.00,10.00,11.00,3.00,14.40,47.00,...,1139.00,3437.00,1.00,0.00,0.00,0.00,4.10,40428.00,36422.00,23.00
3,1,7,AL,BIBB,22005.00,15.00,27.00,7.00,13.00,166.00,...,1919.00,17974.00,0.00,0.00,1.00,0.00,2.50,36892.00,54277.00,20.60
4,1,9,AL,BLOUNT,59512.00,12.00,18.00,0.00,13.30,25.00,...,711.00,3619.00,0.00,1.00,0.00,0.00,2.20,42634.00,52830.00,12.00


In [43]:
ahrf_frame = ahrf_frame.merge(ahrf_frame_2023, how='left', on=['State_FIPS','County_FIPS','State','County'])

In [44]:
[col for col in ahrf_frame.columns if '2022' in col]

['Population Estimate 2022', 'Unemployment Rate, 16+ 2022']

In [45]:
# Determine a set of medical care providers variables to analyze
medical_providers_columns = {"fips_st":"FIPS State Code",
                              "fips_cnty":"FIPS County Code",
                              "st_name_abbrev":"State Name Abbreviation",
                               "cnty_name":"County Name"}

medical_providers_columns['popn_est_23']="Population Estimate 2023"
medical_providers_columns['phys_nf_prim_care_pc_exc_rsdt_22']="Phys,Primary Care, Patient Care Non-Fed 2022"
medical_providers_columns['md_nf_activ_22']="Total Active M.D.s Non-Federal 2022"
medical_providers_columns['do_nf_activ_22']="Total Active D.O.s Non-Federal 2022"

all_columns = medical_providers_columns

all_columns['pers_noins_lt65_pct_21']='% <65 without Health Insurance 2021'


# Determine a set of medical facilities utilization variables to analyze
all_columns['hosp_beds_22'] = 'Hospital Beds 2022'
all_columns['stgh_hosp_beds_22'] = 'Short Term General Hosp Beds 2022'
all_columns['stgh_inpat_dys_22'] = 'Inpatient Days in ST Gen Hosp 2022'
all_columns['hosp_22'] = 'Total Number Hospitals 2022'
all_columns['stgh_22'] = '# Short Term General Hosps 2022'
all_columns['hosp_adm_22'] = 'Hospital Admissions 2022'
all_columns['stgh_hosp_adm_22'] = 'Short Term Gen Hosp Admissions 2022'
all_columns['stgh_medcr_inpat_dys_22'] = 'Total Medicare Inpatient Days Short Term General Hospitals 2022'
all_columns['stgh_00_39_util_rate_pct_22'] = 'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2022'
all_columns['stgh_40_59_util_rate_pct_22'] = 'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2022'
all_columns['stgh_60_79_util_rate_pct_22'] = 'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2022'
all_columns['stgh_80_plus_util_rate_pct_22'] = 'Dist Hosp By 80+% Util Rate Short Term General Hospitals 2022'

# Determine a set of demographics variables to analyze
#all_columns.extend(["% <65 without Health Insurance 2010"])
all_columns['unemply_rate_ge16_23']="Unemployment Rate, 16+ 2023"
all_columns['per_cap_persnl_incom_22']="Per Capita Personal Income 2022"
all_columns['medn_hhi_acs_22']="Median Household Income 2022"
all_columns['pers_povty_pct_22']="Percent Persons in Poverty 2022"


ahrf_frame_2024 = pd.read_csv('../AHRF_project_master/source/AHRF_2023-2024/DATA/ahrf2024.csv')[list(all_columns)]
ahrf_frame_2024 = ahrf_frame_2024.rename(columns=all_columns)
ahrf_frame_2024 = convert_to_numeric(ahrf_frame_2024)
ahrf_frame_2024 = ahrf_frame_2024.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2024['County'] = ahrf_frame_2024['County'].str.upper()
ahrf_frame_2024['State_FIPS'] = ahrf_frame_2024['State_FIPS'].astype(str)
ahrf_frame_2024['County_FIPS'] = ahrf_frame_2024['County_FIPS'].astype(str)

ahrf_frame_2024.head()

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91901/1571832215.py:39: DtypeWarning: Columns (14,16,24) have mixed types. Specify dtype option on import or set low_memory=False.
  ahrf_frame_2024 = pd.read_csv('../AHRF_project_master/source/AHRF_2023-2024/DATA/ahrf2024.csv')[list(all_columns)]


,State_FIPS,County_FIPS,State,County,Population Estimate 2023,"Phys,Primary Care, Patient Care Non-Fed 2022",Total Active M.D.s Non-Federal 2022,Total Active D.O.s Non-Federal 2022,% <65 without Health Insurance 2021,Hospital Beds 2022,...,Short Term Gen Hosp Admissions 2022,Total Medicare Inpatient Days Short Term General Hospitals 2022,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2022,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2022,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2022,Dist Hosp By 80+% Util Rate Short Term General Hospitals 2022,"Unemployment Rate, 16+ 2023",Per Capita Personal Income 2022,Median Household Income 2022,Percent Persons in Poverty 2022
0,1,1,AL,AUTAUGA,60342.00,25.00,43.00,2.00,10.00,77.00,...,3414.00,12510.00,0.00,0.00,1.00,0.00,2.20,49391.00,68315.00,11.80
1,1,3,AL,BALDWIN,253507.00,160.00,494.00,61.00,11.00,418.00,...,20746.00,58316.00,0.00,1.00,2.00,0.00,2.30,56747.00,71039.00,12.40
2,1,5,AL,BARBOUR,24585.00,8.00,11.00,1.00,12.70,47.00,...,1159.00,4582.00,1.00,0.00,0.00,0.00,4.40,40560.00,39712.00,26.70
3,1,7,AL,BIBB,21868.00,21.00,33.00,10.00,11.40,166.00,...,1955.00,17943.00,0.00,0.00,1.00,0.00,2.50,37513.00,50669.00,20.00
4,1,9,AL,BLOUNT,59816.00,12.00,18.00,0.00,12.80,25.00,...,914.00,3652.00,0.00,1.00,0.00,0.00,2.10,43744.00,57440.00,13.60


In [46]:
ahrf_frame = ahrf_frame.merge(ahrf_frame_2024, how='left', on=['State_FIPS','County_FIPS','State','County'])

In [47]:
len([col for col in ahrf_frame.columns if '2025' in col])

0

In [48]:
# Determine a set of medical care providers variables to analyze
medical_providers_columns = {"fips_st":"FIPS State Code",
                              "fips_cnty":"FIPS County Code",
                              "st_name_abbrev":"State Name Abbreviation",
                               "cnty_name":"County Name"}

medical_providers_columns['popn_est_24']="Population Estimate 2024"
medical_providers_columns['phys_nf_prim_care_pc_exc_rsdt_23']="Phys,Primary Care, Patient Care Non-Fed 2023"
medical_providers_columns['md_nf_activ_23']="Total Active M.D.s Non-Federal 2023"
medical_providers_columns['do_nf_activ_23']="Total Active D.O.s Non-Federal 2023"

all_columns = medical_providers_columns

all_columns['pers_noins_lt65_pct_22']='% <65 without Health Insurance 2022'


# Determine a set of medical facilities utilization variables to analyze
all_columns['hosp_beds_23'] = 'Hospital Beds 2023'
all_columns['stgh_hosp_beds_23'] = 'Short Term General Hosp Beds 2023'
all_columns['stgh_inpat_dys_23'] = 'Inpatient Days in ST Gen Hosp 2023'
all_columns['hosp_23'] = 'Total Number Hospitals 2023'
all_columns['stgh_23'] = '# Short Term General Hosps 2023'
all_columns['hosp_adm_23'] = 'Hospital Admissions 2023'
all_columns['stgh_hosp_adm_23'] = 'Short Term Gen Hosp Admissions 2023'
all_columns['stgh_medcr_inpat_dys_23'] = 'Total Medicare Inpatient Days Short Term General Hospitals 2023'
all_columns['stgh_00_39_util_rate_pct_23'] = 'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2023'
all_columns['stgh_40_59_util_rate_pct_23'] = 'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2023'
all_columns['stgh_60_79_util_rate_pct_23'] = 'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2023'
all_columns['stgh_80_plus_util_rate_pct_23'] = 'Dist Hosp By 80+% Util Rate Short Term General Hospitals 2023'

# Determine a set of demographics variables to analyze
#all_columns.extend(["% <65 without Health Insurance 2010"])
all_columns['unemply_rate_ge16_24']="Unemployment Rate, 16+ 2024"
all_columns['per_cap_persnl_incom_23']="Per Capita Personal Income 2023"
all_columns['medn_hhi_acs_23']="Median Household Income 2023"
all_columns['pers_povty_pct_23']="Percent Persons in Poverty 2023"


ahrf_frame_2025 = pd.read_csv('../AHRF_project_master/source/AHRF_2024-2025/DATA/AHRF2025.csv')[list(all_columns)]
ahrf_frame_2025 = ahrf_frame_2025.rename(columns=all_columns)
ahrf_frame_2025 = convert_to_numeric(ahrf_frame_2025)
ahrf_frame_2025 = ahrf_frame_2025.rename(columns = {"County Name":"County","State Name Abbreviation":"State","FIPS County Code":"County_FIPS","FIPS State Code":"State_FIPS"})
ahrf_frame_2025['County'] = ahrf_frame_2025['County'].str.upper()
ahrf_frame_2025['State_FIPS'] = ahrf_frame_2025['State_FIPS'].astype(str)
ahrf_frame_2025['County_FIPS'] = ahrf_frame_2025['County_FIPS'].astype(str)

ahrf_frame_2025.head()

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91901/4072185085.py:39: DtypeWarning: Columns (14,16,24) have mixed types. Specify dtype option on import or set low_memory=False.
  ahrf_frame_2025 = pd.read_csv('../AHRF_project_master/source/AHRF_2024-2025/DATA/AHRF2025.csv')[list(all_columns)]


,State_FIPS,County_FIPS,State,County,Population Estimate 2024,"Phys,Primary Care, Patient Care Non-Fed 2023",Total Active M.D.s Non-Federal 2023,Total Active D.O.s Non-Federal 2023,% <65 without Health Insurance 2022,Hospital Beds 2023,...,Short Term Gen Hosp Admissions 2023,Total Medicare Inpatient Days Short Term General Hospitals 2023,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2023,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2023,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2023,Dist Hosp By 80+% Util Rate Short Term General Hospitals 2023,"Unemployment Rate, 16+ 2024",Per Capita Personal Income 2023,Median Household Income 2023,Percent Persons in Poverty 2023
0,1,1,AL,AUTAUGA,61464.00,22.00,41.00,1.00,8.20,71.00,...,3159.00,11720.00,0.00,0.00,1.00,0.00,2.70,53079.00,69841.00,11.70
1,1,3,AL,BALDWIN,261608.00,172.00,523.00,71.00,10.20,546.00,...,21501.00,57419.00,1.00,1.00,1.00,0.00,2.70,60969.00,75019.00,10.00
2,1,5,AL,BARBOUR,24358.00,7.00,12.00,1.00,12.10,47.00,...,1100.00,3153.00,1.00,0.00,0.00,0.00,4.30,41531.00,44290.00,25.50
3,1,7,AL,BIBB,22258.00,26.00,38.00,16.00,10.80,166.00,...,962.00,17027.00,0.00,0.00,1.00,0.00,3.00,39835.00,51215.00,19.40
4,1,9,AL,BLOUNT,60163.00,10.00,16.00,0.00,12.50,25.00,...,792.00,2471.00,0.00,1.00,0.00,0.00,2.70,45021.00,61096.00,12.80


In [49]:
ahrf_frame = ahrf_frame.merge(ahrf_frame_2025, how='left', on=['State_FIPS','County_FIPS','State','County'])

In [50]:
[col for col in ahrf_frame.columns if '2022' in col]

['Population Estimate 2022',
 'Unemployment Rate, 16+ 2022',
 'Phys,Primary Care, Patient Care Non-Fed 2022',
 'Total Active M.D.s Non-Federal 2022',
 'Total Active D.O.s Non-Federal 2022',
 'Hospital Beds 2022',
 'Short Term General Hosp Beds 2022',
 'Inpatient Days in ST Gen Hosp 2022',
 'Total Number Hospitals 2022',
 '# Short Term General Hosps 2022',
 'Hospital Admissions 2022',
 'Short Term Gen Hosp Admissions 2022',
 'Total Medicare Inpatient Days Short Term General Hospitals 2022',
 'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2022',
 'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2022',
 'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2022',
 'Dist Hosp By 80+% Util Rate Short Term General Hospitals 2022',
 'Per Capita Personal Income 2022',
 'Median Household Income 2022',
 'Percent Persons in Poverty 2022',
 '% <65 without Health Insurance 2022']

In [51]:
ahrf_frame

,State_FIPS,County_FIPS,State,County,Population Estimate 2010,Population Estimate 2020,Population Estimate 2011,Population Estimate 2012,Population Estimate 2013,Population Estimate 2014,...,Short Term Gen Hosp Admissions 2023,Total Medicare Inpatient Days Short Term General Hospitals 2023,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2023,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2023,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2023,Dist Hosp By 80+% Util Rate Short Term General Hospitals 2023,"Unemployment Rate, 16+ 2024",Per Capita Personal Income 2023,Median Household Income 2023,Percent Persons in Poverty 2023
0,22,113,LA,VERMILION,57999.00,57359.00,58276.00,58723.00,59253.00,59616.00,...,2664.00,6624.00,0.00,2.00,0.00,0.00,4.10,49563.00,57537.00,17.50
1,27,053,MN,HENNEPIN,1152425.00,1281565.00,1168431.00,1184576.00,1198778.00,1212064.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,04,013,AZ,MARICOPA,3817117.00,4420568.00,3880244.00,3942169.00,4009412.00,4087191.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,35,006,NM,CIBOLA,27213.00,27172.00,27658.00,27334.00,27335.00,27349.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,48,201,TX,HARRIS,4092459.00,4731145.00,4180894.00,4253700.00,4336853.00,4441370.00,...,464521.00,1167479.00,6.00,6.00,13.00,4.00,4.40,73862.00,73104.00,16.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,01,043,AL,CULLMAN,80406.00,87866.00,80536.00,80440.00,80811.00,81289.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
761,36,065,NY,ONEIDA,234878.00,232125.00,234287.00,233556.00,233585.00,232871.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
762,37,197,NC,YADKIN,38406.00,37214.00,38279.00,38084.00,38038.00,37792.00,...,0.00,0.00,0.00,0.00,0.00,0.00,3.10,49886.00,60321.00,12.90
763,53,077,WA,YAKIMA,243231.00,256728.00,247141.00,246977.00,247044.00,247687.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
# Stripping out double spaces in 2013 columns
ahrf_frame.columns = ahrf_frame.columns.str.replace(r'\s+', ' ', regex=True)

In [53]:
# Standardizing column names
ahrf_frame = ahrf_frame.rename(columns={'Total Medicare Inpatient Days (ST Gen Hosps) 2011':'Total Medicare Inpatient Days Short Term General Hospitals 2011',
                                       'Total Medicare Inpatient Days (ST Gen Hosps) 2012':'Total Medicare Inpatient Days Short Term General Hospitals 2012',
                                       'Dist Hosp By 00 - 39% Util Rate STGEN 2011':'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2011',
                                       'Dist Hosp By 00 - 39% Util Rate STGEN 2012':'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2012',
                                        'Dist Hosp By 40 - 59% Util Rate STGEN 2011':'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2011',
                                       'Dist Hosp By 40 - 59% Util Rate STGEN 2012':'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2012',
                                        'Dist Hosp By 60 - 79% Util Rate STGEN 2011':'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2011',
                                        'Dist Hosp By 60 - 79% Util Rate STGEN 2012':'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2012',
                                        'Dist Hosp By 80+% Util Rate STGEN 2011': 'Dist Hosp By 80+% Util Rate Short Term General Hospitals 2011',
                                        'Dist Hosp By 80+% Util Rate STGEN 2012': 'Dist Hosp By 80+% Util Rate Short Term General Hospitals 2012'
                                       })


In [54]:
len(ahrf_frame.columns)

287

In [55]:
# Normalizing count columns to be per capita
for year in range(2010,2026):
    populationcol = f'Population Estimate {year}'
    columns= [f'Phys,Primary Care, Patient Care Non-Fed {year}',
 f'Total Active M.D.s Non-Federal {year}',
 f'Total Active D.O.s Non-Federal {year}',
 f'Hospital Beds {year}',
 f'Short Term General Hosp Beds {year}',
 f'Inpatient Days in ST Gen Hosp {year}',
 f'Total Number Hospitals {year}',
 f'# Short Term General Hosps {year}',
 f'Hospital Admissions {year}',
 f'Short Term Gen Hosp Admissions {year}',
 f'Total Medicare Inpatient Days Short Term General Hospitals {year}']
    for col in columns:
        if populationcol in ahrf_frame.columns and col in ahrf_frame.columns:
            ahrf_frame["Per Capita " + col] = ahrf_frame[col] / ahrf_frame[populationcol] 
            ahrf_frame = ahrf_frame.drop(columns=col)
        elif populationcol not in ahrf_frame.columns and col in ahrf_frame.columns:
            ahrf_frame["Per Capita " + col] = np.nan
            ahrf_frame = ahrf_frame.drop(columns=col)
        else:
            ahrf_frame["Per Capita " + col] = np.nan

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91901/583847565.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ahrf_frame["Per Capita " + col] = np.nan
/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91901/583847565.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ahrf_frame["Per Capita " + col] = ahrf_frame[col] / ahrf_frame[populationcol]
/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91901/583847565.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resu

In [56]:
[col for col in ahrf_frame.columns if '2015' in col]

['Population Estimate 2015',
 'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2015',
 'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2015',
 'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2015',
 'Dist Hosp By 80+% Util Rate Short Term General Hospitals 2015',
 '% <65 without Health Insurance 2015',
 'Unemployment Rate, 16+ 2015',
 'Per Capita Personal Income 2015',
 'Median Household Income 2015',
 'Percent Persons in Poverty 2015',
 'Per Capita Phys,Primary Care, Patient Care Non-Fed 2015',
 'Per Capita Total Active M.D.s Non-Federal 2015',
 'Per Capita Total Active D.O.s Non-Federal 2015',
 'Per Capita Hospital Beds 2015',
 'Per Capita Short Term General Hosp Beds 2015',
 'Per Capita Inpatient Days in ST Gen Hosp 2015',
 'Per Capita Total Number Hospitals 2015',
 'Per Capita # Short Term General Hosps 2015',
 'Per Capita Hospital Admissions 2015',
 'Per Capita Short Term Gen Hosp Admissions 2015',
 'Per Capita Total Medicare Inpatient Days

In [57]:
# Ensuring all other columns have a column for every year
for year in range(2010,2026):
    columns= [f'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals {year}',
 f'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals {year}',
 f'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals {year}',
 f'Dist Hosp By 80+% Util Rate Short Term General Hospitals {year}',
 f'% <65 without Health Insurance {year}',
 f'Unemployment Rate, 16+ {year}',
 f'Per Capita Personal Income {year}',
 f'Median Household Income {year}',
 f'Percent Persons in Poverty {year}']
    for col in columns:
        if col not in ahrf_frame.columns:
            ahrf_frame[col] = np.nan

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91901/3545745636.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ahrf_frame[col] = np.nan
/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91901/3545745636.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ahrf_frame[col] = np.nan
/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91901/3545745636.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [58]:
len(ahrf_frame.columns)

340

In [59]:
ahrf_frame.columns

Index(['State_FIPS', 'County_FIPS', 'State', 'County',
       'Population Estimate 2010', 'Population Estimate 2020',
       'Population Estimate 2011', 'Population Estimate 2012',
       'Population Estimate 2013', 'Population Estimate 2014',
       ...
       'Percent Persons in Poverty 2024',
       'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2025',
       'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2025',
       'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2025',
       'Dist Hosp By 80+% Util Rate Short Term General Hospitals 2025',
       '% <65 without Health Insurance 2025', 'Unemployment Rate, 16+ 2025',
       'Per Capita Personal Income 2025', 'Median Household Income 2025',
       'Percent Persons in Poverty 2025'],
      dtype='object', length=340)

In [60]:
test = ahrf_frame[[col for col in ahrf_frame.columns if 'Short Term Gen Hosp Admissions' in col or 'FIPS' in col or 'Dist Hosp By 00 - 39% Util Rate Short Term General' in col]]
test

,State_FIPS,County_FIPS,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2020,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2015,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2010,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2011,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2012,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2013,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2014,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2016,...,Per Capita Short Term Gen Hosp Admissions 2019,Per Capita Short Term Gen Hosp Admissions 2020,Per Capita Short Term Gen Hosp Admissions 2021,Per Capita Short Term Gen Hosp Admissions 2022,Per Capita Short Term Gen Hosp Admissions 2023,Per Capita Short Term Gen Hosp Admissions 2024,Per Capita Short Term Gen Hosp Admissions 2025,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2018,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2024,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2025
0,22,113,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,0.05,0.05,0.05,0.05,0.05,NaN,NaN,NaN,NaN,NaN
1,27,053,0.00,0.00,2.00,0.00,0.00,0.00,0.00,0.00,...,0.13,0.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,04,013,1.00,2.00,1.00,1.00,2.00,3.00,2.00,2.00,...,0.09,0.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,35,006,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,...,0.04,0.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,48,201,6.00,3.00,8.00,12.00,9.00,7.00,5.00,4.00,...,0.10,0.09,0.09,0.10,0.10,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,01,043,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.09,0.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
761,36,065,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.12,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
762,37,197,0.00,0.00,1.00,1.00,1.00,1.00,1.00,0.00,...,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN
763,53,077,1.00,2.00,2.00,1.00,1.00,1.00,2.00,2.00,...,0.06,0.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
[col for col in ahrf_frame.columns if '2018' in col]

['Population Estimate 2018',
 '% <65 without Health Insurance 2018',
 'Unemployment Rate, 16+ 2018',
 'Per Capita Personal Income 2018',
 'Median Household Income 2018',
 'Percent Persons in Poverty 2018',
 'Per Capita Phys,Primary Care, Patient Care Non-Fed 2018',
 'Per Capita Total Active M.D.s Non-Federal 2018',
 'Per Capita Total Active D.O.s Non-Federal 2018',
 'Per Capita Hospital Beds 2018',
 'Per Capita Short Term General Hosp Beds 2018',
 'Per Capita Inpatient Days in ST Gen Hosp 2018',
 'Per Capita Total Number Hospitals 2018',
 'Per Capita # Short Term General Hosps 2018',
 'Per Capita Hospital Admissions 2018',
 'Per Capita Short Term Gen Hosp Admissions 2018',
 'Per Capita Total Medicare Inpatient Days Short Term General Hospitals 2018',
 'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals 2018',
 'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals 2018',
 'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals 2018',
 'Dist Hosp By 80+% Util Rate

In [62]:
id_cols = ['State','County','State_FIPS','County_FIPS']

long_df = ahrf_frame.melt(
   id_vars=id_cols,
   var_name="variable",
   value_name="value"
)

In [63]:
id_cols = ['State','County','State_FIPS','County_FIPS']

# all measure columns that end with a year
value_cols = [c for c in ahrf_frame.columns if c not in id_cols]

long = ahrf_frame.melt(
   id_vars=id_cols,
   value_vars=value_cols,
   var_name="metric_year",
   value_name="value"
)

# split "Percent Persons in Poverty 2018" -> metric="Percent Persons in Poverty", year=2018
long["year"] = pd.to_numeric(long["metric_year"].str.strip().str.extract(r"(\d{4})$")[0], errors='coerce')
long["metric"] = long["metric_year"].str.replace(r"\s+\d{4}$", "", regex=True)


In [64]:
long

,State,County,State_FIPS,County_FIPS,metric_year,value,year,metric
0,LA,VERMILION,22,113,Population Estimate 2010,57999.00,2010,Population Estimate
1,MN,HENNEPIN,27,053,Population Estimate 2010,1152425.00,2010,Population Estimate
2,AZ,MARICOPA,04,013,Population Estimate 2010,3817117.00,2010,Population Estimate
3,NM,CIBOLA,35,006,Population Estimate 2010,27213.00,2010,Population Estimate
4,TX,HARRIS,48,201,Population Estimate 2010,4092459.00,2010,Population Estimate
...,...,...,...,...,...,...,...,...
257035,AL,CULLMAN,01,043,Percent Persons in Poverty 2025,NaN,2025,Percent Persons in Poverty
257036,NY,ONEIDA,36,065,Percent Persons in Poverty 2025,NaN,2025,Percent Persons in Poverty
257037,NC,YADKIN,37,197,Percent Persons in Poverty 2025,NaN,2025,Percent Persons in Poverty
257038,WA,YAKIMA,53,077,Percent Persons in Poverty 2025,NaN,2025,Percent Persons in Poverty


In [65]:
long[long['year'].isna()]['metric_year'].unique()

array([], dtype=object)

In [66]:
# now pivot so each metric is a column again
out = (
   long.pivot_table(
       index=id_cols + ["year"],
       columns="metric",
       values="value",
       aggfunc="first"
   )
   .reset_index()
)

# flatten the column index created by pivot_table
out.columns.name = None

In [67]:
out

,State,County,State_FIPS,County_FIPS,year,% <65 without Health Insurance,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals,Dist Hosp By 80+% Util Rate Short Term General Hospitals,...,"Per Capita Phys,Primary Care, Patient Care Non-Fed",Per Capita Short Term Gen Hosp Admissions,Per Capita Short Term General Hosp Beds,Per Capita Total Active D.O.s Non-Federal,Per Capita Total Active M.D.s Non-Federal,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,Per Capita Total Number Hospitals,Percent Persons in Poverty,Population Estimate,"Unemployment Rate, 16+"
0,AK,ANCHORAGE (B),02,020,2010,18.40,0.00,2.00,2.00,0.00,...,0.00,0.12,0.00,0.00,0.00,0.16,0.00,9.60,291826.00,6.80
1,AK,ANCHORAGE (B),02,020,2011,18.30,0.00,2.00,2.00,0.00,...,0.00,0.12,0.00,0.00,0.00,0.16,0.00,8.50,295570.00,6.10
2,AK,ANCHORAGE (B),02,020,2012,19.80,0.00,2.00,2.00,0.00,...,0.00,0.12,0.00,0.00,0.00,0.20,0.00,9.00,298610.00,5.40
3,AK,ANCHORAGE (B),02,020,2013,18.10,1.00,0.00,3.00,0.00,...,0.00,0.12,0.00,0.00,0.00,0.28,0.00,7.70,300950.00,NaN
4,AK,ANCHORAGE (B),02,020,2014,16.70,0.00,1.00,3.00,0.00,...,0.00,0.12,0.00,0.00,0.00,0.22,0.00,10.00,301010.00,5.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9511,WY,NATRONA,56,025,2017,14.20,1.00,1.00,0.00,0.00,...,0.00,0.12,0.00,0.00,0.00,0.21,0.00,11.10,79547.00,5.30
9512,WY,NATRONA,56,025,2018,13.20,NaN,NaN,NaN,NaN,...,0.00,NaN,NaN,0.00,0.00,NaN,NaN,9.90,79115.00,4.60
9513,WY,NATRONA,56,025,2019,14.50,1.00,1.00,0.00,0.00,...,0.00,0.12,0.00,0.00,0.00,0.23,0.00,9.90,79858.00,3.90
9514,WY,NATRONA,56,025,2020,NaN,1.00,1.00,0.00,0.00,...,0.00,0.10,0.00,0.00,0.00,0.22,0.00,9.40,79955.00,7.80


In [68]:
out[out['year'].isna()]

,State,County,State_FIPS,County_FIPS,year,% <65 without Health Insurance,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals,Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals,Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals,Dist Hosp By 80+% Util Rate Short Term General Hospitals,...,"Per Capita Phys,Primary Care, Patient Care Non-Fed",Per Capita Short Term Gen Hosp Admissions,Per Capita Short Term General Hosp Beds,Per Capita Total Active D.O.s Non-Federal,Per Capita Total Active M.D.s Non-Federal,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,Per Capita Total Number Hospitals,Percent Persons in Poverty,Population Estimate,"Unemployment Rate, 16+"


In [69]:
out = out.rename(columns={'year':'Year'})

In [70]:
out.to_csv('../data/ARHF_Full.csv', index=False)